In [ ]:
!pip install -q google-genai pypdf scikit-learn

In [ ]:
from google import genai
from google.colab import userdata
from google.colab import files
from pypdf import PdfReader

In [ ]:
uploaded = files.upload()

Saving Sunridge Institute of Technology — Student Policy Handbook 2025–26.pdf to Sunridge Institute of Technology — Student Policy Handbook 2025–26.pdf


In [ ]:

pdf_file = list(uploaded.keys())[0]


reader = PdfReader(pdf_file)

print("Total Pages:", len(reader.pages))

Total Pages: 6


In [ ]:
pdf_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text

print("Total Characters:", len(pdf_text))

Total Characters: 11204


In [ ]:
print(pdf_text[:1000])

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Technology, Devgarh
Campus. It covers admissions, the refund of fees, hostel residence, examinations and
re-evaluation, library usage, attendance, scholarships and the student grievance
procedure. In the event of any dispute, the version of this handbook published on the
Registrar's notice board on 1 July 2025 shall be treated as final. Students are expected
to read this handbook in full within the first two weeks of the semester.
1. Admission Rules
Admission to all undergraduate programmes at Sunridge Institute of Technology is
made strictly through the Sunridge Common Admission Test (SCAT), conducted once a
year in the third week of May. A candidate must secure a minimum SCAT score of 142
out of 300 to be eligible for c

In [ ]:
chunk_size = 3000
overlap = 150

chunks = []

start = 0

while start < len(pdf_text):
    end = start + chunk_size
    chunk = pdf_text[start:end]
    chunks.append(chunk)

    start += chunk_size - overlap

print("Total Chunks:", len(chunks))

Total Chunks: 4


In [ ]:
print("Chunk 1:\n")
print(chunks[0])

Chunk 1:

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Technology, Devgarh
Campus. It covers admissions, the refund of fees, hostel residence, examinations and
re-evaluation, library usage, attendance, scholarships and the student grievance
procedure. In the event of any dispute, the version of this handbook published on the
Registrar's notice board on 1 July 2025 shall be treated as final. Students are expected
to read this handbook in full within the first two weeks of the semester.
1. Admission Rules
Admission to all undergraduate programmes at Sunridge Institute of Technology is
made strictly through the Sunridge Common Admission Test (SCAT), conducted once a
year in the third week of May. A candidate must secure a minimum SCAT score of 142
out of 300 to be elig

In [ ]:
from google import genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

print("✅ Gemini connected successfully!")

✅ Gemini connected successfully!


In [ ]:
embeddings = []

for i, chunk in enumerate(chunks):
    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=chunk
    )

    embeddings.append(response.embeddings[0].values)

print("Total Embeddings:", len(embeddings))

Total Embeddings: 4


In [ ]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [ ]:
chunk_data = []

for chunk, embedding in zip(chunks, embeddings):
    chunk_data.append({
        "chunk": chunk,
        "embedding": embedding
    })

print("Stored", len(chunk_data), "chunks with embeddings.")

Stored 4 chunks with embeddings.


In [ ]:
print("First 200 characters of Chunk 1:")
print(chunk_data[0]["chunk"][:200])

print("\nEmbedding length:")
print(len(chunk_data[0]["embedding"]))

First 200 characters of Chunk 1:
Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlemen

Embedding length:
3072


In [ ]:
import numpy as np

In [ ]:
def search_chunks(question, top_k=3):
    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=question
    )

    question_embedding = np.array(response.embeddings[0].values)

    similarities = []


    for i, embedding in enumerate(embeddings):
        chunk_embedding = np.array(embedding)

        similarity = np.dot(question_embedding, chunk_embedding) / (
            np.linalg.norm(question_embedding) *
            np.linalg.norm(chunk_embedding)
        )

        similarities.append((i, similarity))

    similarities.sort(key=lambda x: x[1], reverse=True)

    return similarities[:top_k]

In [ ]:
question = "What is the attendance requirement?"

results = search_chunks(question)

for index, score in results:
    print("=" * 60)
    print("Chunk Number:", index + 1)
    print("Similarity Score:", round(score, 4))
    print()
    print(chunks[index])
    print()

Chunk Number: 14
Similarity Score: 0.6232

sued for
borrowing and may be consulted only inside the reading hall. Marking, underlining or
folding pages of library books is treated as damage and fined at ₹300 per book. The
library conducts its annual stock verification in the last week of April, during which all
borrowed books must be returned regardless of due dates.
6. Attendance and Leave
A minimum attendance of 75 percent in each course is mandatory for appearing in the
end-semester examination of that course. Attendance is computed course-wise, not as
an aggregate. A student whose attendance in a course falls between 65 and 75 percent
may apply for condonation on medical grounds only, supported by a medical
certificate, on payment of a condonation fee of ₹800 per course; condonation below 65
percent is not granted by any authori

Chunk Number: 10
Similarity Score: 0.6155

ation, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues.

In [ ]:
def ask_rag(question):

    results = search_chunks(question)

    context = ""

    print("Top Retrieved Chunks:\n")

    for index, score in results:
        print("=" * 60)
        print(f"Chunk {index + 1}")
        print(f"Similarity Score: {score:.4f}\n")
        print(chunks[index][:300])  # Show first 300 characters
        print()

        context += chunks[index] + "\n\n"

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the information given below.

If the answer is not found in the document, reply exactly:

"I cannot find this information in the document."

Document:
{context}

Question:
{question}
"""

    response = client.models.generate_content(
        model="models/gemini-3.5-flash",
        contents=prompt
    )

    print("\n" + "=" * 60)
    print("Final Answer:\n")
    print(response.text)

In [ ]:
ask_rag("What is the minimum attendance required?")

Top Retrieved Chunks:

Chunk 4
Similarity Score: 0.6030

g pages of library books is treated as damage and fined at ₹300 per book. The
library conducts its annual stock verification in the last week of April, during which all
borrowed books must be returned regardless of due dates.
6. Attendance and Leave
A minimum attendance of 75 percent in each course 

Chunk 3
Similarity Score: 0.5903

ions and Re-evaluation Procedure
End-semester examinations are held twice a year, in December and in May. To be
eligible to sit an end-semester examination, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A student
who misses an examination o

Chunk 1
Similarity Score: 0.5527

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Te

In [ ]:
ask_rag("Who is the Prime Minister of India?")

Top Retrieved Chunks:

Chunk 3
Similarity Score: 0.5095

ions and Re-evaluation Procedure
End-semester examinations are held twice a year, in December and in May. To be
eligible to sit an end-semester examination, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A student
who misses an examination o

Chunk 2
Similarity Score: 0.5054

 after deducting 10 percent of the total fees paid as
administrative charges. The admission confirmation fee of ₹18,500 is fully non-
refundable in all cases.
If the cancellation request is received within 15 days after the commencement of
classes, the deduction rises to 25 percent of the total fees

Chunk 1
Similarity Score: 0.4966

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Te

In [ ]:
ask_rag("What is the minimum attendance percentage required?")

Top Retrieved Chunks:

Chunk 10
Similarity Score: 0.6598

ation, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A student
who misses an examination on medical grounds may apply for a make-up examination
within 10 days, attaching a medical certificate issued by a registered practitioner;
make-up exa

Chunk 14
Similarity Score: 0.6547

sued for
borrowing and may be consulted only inside the reading hall. Marking, underlining or
folding pages of library books is treated as damage and fined at ₹300 per book. The
library conducts its annual stock verification in the last week of April, during which all
borrowed books must be returned

Chunk 9
Similarity Score: 0.6222

d 7:00 PM and are not
permitted inside residential corridors.
The hostel mess operates on a compulsory meal plan. Mess rebate for absence is
granted only for continuous absences of 7 days or longer, applied for in advance on
Form HM-3, at the rate of ₹95 per day. Raggi

In [ ]:
ask_rag("What is the re-evaluation fee?")

Top Retrieved Chunks:

Chunk 11
Similarity Score: 0.7448

e for laboratory courses, project work, or viva-voce
examinations.
Under re-evaluation, the answer script is assessed independently by a second examiner
who has no access to the first examiner's marks. If the revised total differs from the
original by 12 marks or more (out of 100), the script is sen

Chunk 10
Similarity Score: 0.6920

ation, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A student
who misses an examination on medical grounds may apply for a make-up examination
within 10 days, attaching a medical certificate issued by a registered practitioner;
make-up exa

Chunk 12
Similarity Score: 0.6510

nscript.
Re-evaluation results are declared within 30 days of the application deadline. A student
may apply for re-evaluation in a maximum of 3 papers per semester. Use of unfairmeans in any examination leads to cancellation of that paper and a hearing before the
Exam

In [ ]:
ask_rag("Who invented Python?")

Top Retrieved Chunks:

Chunk 4
Similarity Score: 0.4472

g pages of library books is treated as damage and fined at ₹300 per book. The
library conducts its annual stock verification in the last week of April, during which all
borrowed books must be returned regardless of due dates.
6. Attendance and Leave
A minimum attendance of 75 percent in each course 

Chunk 1
Similarity Score: 0.4388

Sunridge Institute of Technology
Student Policy Handbook · Academic Year 2025–26 · Issued by the Office of the Registrar
This handbook is the official statement of the rules, procedures and entitlements that
apply to every student enrolled at the Sunridge Institute of Technology, Devgarh
Campus. It 

Chunk 3
Similarity Score: 0.4368

ions and Re-evaluation Procedure
End-semester examinations are held twice a year, in December and in May. To be
eligible to sit an end-semester examination, a student must have a minimum
attendance of 75 percent in that course and must have cleared all fee dues. A studen

Experiment Observation

I changed the chunk size from 800 to 3000 characters. The total number of chunks decreased because each chunk contained more text. The answers had more context but sometimes included extra unrelated information. With 800-character chunks, the retrieval was more focused, while 3000-character chunks provided broader context.